# Train on Colab
Runtime > Change runtime type > GPU (T4) before running.

In [ ]:
# 1. Clone (or update) the repo
import os
REPO = 'https://github.com/ahmed-khaled04/bird_image_to_caption.git'
ROOT = '/content/bird_image_to_caption'
if not os.path.exists(ROOT):
    !git clone {REPO} {ROOT}
else:
    !git -C {ROOT} pull
%cd {ROOT}

In [ ]:
# 2. Dependencies (torch/torchvision are already on Colab, so they are skipped)
!grep -vi -E '^(torch|torchvision)$' requirements.txt > /tmp/req.txt
!pip install -q -r /tmp/req.txt
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3. Keep the trained weights on Drive so they survive the session.
# Only models/ is linked: outputs/ holds tracked files (e.g. tables/duplicate_pairs.csv, needed by `clean`).
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/bird_caption_generator'
os.makedirs(f'{DRIVE}/models', exist_ok=True)
!rm -rf {ROOT}/models
!ln -s {DRIVE}/models {ROOT}/models
# restore anything an earlier version of this cell deleted from the repo
!git -C {ROOT} checkout -- outputs
assert os.path.exists(f'{ROOT}/outputs/tables/duplicate_pairs.csv')

In [ ]:
# 4. Make `import src...` work and build the cleaned data (data/ is not in git)
import sys
sys.path.insert(0, ROOT)
!python -m src.cli clean --no-plots
assert os.path.exists('data/processed'), 'clean failed: read the error above'

In [ ]:
# 5. Settings: change these, then run the next cell
from pathlib import Path

CONFIG = 'configs/transfer_resnet18.yaml'
RUNS = ['resnet18_frozen_adam', 'resnet50_frozen_adam']   # names from the YAML; use one name to train just that model
EPOCHS = None                       # None keeps the YAML value; an int overrides it for every run
LIMIT = None                        # e.g. 64 for a quick check on 64 images per split (saved under outputs/dry_run)

TABLE_DIR = Path('outputs/dry_run' if LIMIT else 'outputs/tables_r18')
FIG_DIR = Path('outputs/dry_run' if LIMIT else 'outputs/figures_r18')


In [ ]:
# 6. Train with the project functions (no CLI)
import yaml
from src.experiment import load_processed, train_run
from src.training.checkpoints import checkpoint_directory
from src.transfer import report, run_and_save, run_configs

config = yaml.safe_load(Path(CONFIG).read_text())
if EPOCHS:
    config['defaults']['epochs'] = EPOCHS

data = load_processed(Path('data/processed'), LIMIT)
train_fn = lambda run, **kwargs: train_run(run, data, **kwargs)   # same wrapper the CLI builds
runs_dir = TABLE_DIR / 'transfer_runs'

for run in run_configs(config, RUNS):
    result = run_and_save(run, train_fn, runs_dir, checkpoint_directory())
    print(run['name'], 'best val acc', round(result['best_val_accuracy'], 4), 'at epoch', result['best_epoch'])

FIG_DIR.mkdir(parents=True, exist_ok=True)
table = report(config, TABLE_DIR, FIG_DIR, runs_dir)


In [ ]:
# 7. Look at the learning curves
from IPython.display import Image, display
display(Image(str(FIG_DIR / 'transfer_learning_curves.png')))
